# Task 5: Validate ALA-specific Species and Label Mapping after Dataset Merge


## Main goals
- Review the ALA portion of the merged dataset.
- Cross-check ALA species names against the main manifest.
- Check whether labels and folders are compatible with the training pipeline.
- Flag naming or mapping inconsistencies.
- Generate a final dataset alignment confirmation note.

## Step 1: Upload the dataset zip

Upload `task5_ala_dataset.zip` when asked.  


In [1]:
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded.keys()))
print("Uploaded:", zip_name)

Saving task5_ala_dataset.zip to task5_ala_dataset.zip
Uploaded: task5_ala_dataset.zip


## Step 2: Extract the dataset

In [2]:
import os, zipfile, re, shutil, hashlib
from pathlib import Path
import pandas as pd

extract_dir = Path("/content/task5_ala_review")
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_dir)

print("Extracted to:", extract_dir)
for p in extract_dir.iterdir():
    print("-", p.name)

Extracted to: /content/task5_ala_review
- task5_ala_merged_dataset


## Step 3: Locate required files and folders

In [3]:
def find_path(root, target_name):
    root = Path(root)
    for p in root.rglob("*"):
        if p.name == target_name:
            return p
    return None

dataset_root = find_path(extract_dir, "task5_ala_merged_dataset")
ala_dir = find_path(extract_dir, "ala_portion")
manifest_path = find_path(extract_dir, "species_manifest.csv")
mapping_path = find_path(extract_dir, "ala_expected_mapping.csv")

print("Dataset root:", dataset_root)
print("ALA folder:", ala_dir)
print("Manifest:", manifest_path)
print("ALA mapping:", mapping_path)

Dataset root: /content/task5_ala_review/task5_ala_merged_dataset
ALA folder: /content/task5_ala_review/task5_ala_merged_dataset/ala_portion
Manifest: /content/task5_ala_review/task5_ala_merged_dataset/species_manifest.csv
ALA mapping: /content/task5_ala_review/task5_ala_merged_dataset/ala_expected_mapping.csv


## Step 4: Load manifest and expected ALA mapping

In [4]:
manifest = pd.read_csv(manifest_path)
mapping = pd.read_csv(mapping_path)

print("Main manifest preview:")
display(manifest.head())

print("ALA expected mapping preview:")
display(mapping.head())

Main manifest preview:


,file_path,species_label,common_name,source,file_type
0,merged_dataset/australian_magpie/australian_ma...,australian_magpie,Australian Magpie,engine_existing,wav
1,merged_dataset/australian_magpie/australian_ma...,australian_magpie,Australian Magpie,engine_existing,wav
2,merged_dataset/australian_magpie/australian_ma...,australian_magpie,Australian Magpie,engine_existing,wav
3,merged_dataset/australian_magpie/australian_ma...,australian_magpie,Australian Magpie,engine_existing,wav
4,merged_dataset/australian_magpie/australian_ma...,australian_magpie,Australian Magpie,engine_existing,wav


ALA expected mapping preview:


,ala_species_name,expected_project_label,notes
0,Australian Magpie,australian_magpie,NaN
1,Laughing Kookaburra,laughing_kookaburra,NaN
2,Rainbow-Lorikeet,rainbow_lorikeet,NaN
3,Sulphur Crested Cockatoo,sulphur_crested_cockatoo,NaN
4,Superb Fairy Wren,superb_fairy_wren,NaN


## Step 5: Define label normalisation function

In [5]:
def normalise_label(name):
    """Convert species names into project-style labels.

    Example:
    'Rainbow-Lorikeet' -> 'rainbow_lorikeet'
    'Sulphur Crested Cockatoo' -> 'sulphur_crested_cockatoo'
    """
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

mapping["normalised_ala_label"] = mapping["ala_species_name"].apply(normalise_label)
mapping["mapping_matches_expected"] = mapping["normalised_ala_label"] == mapping["expected_project_label"]

display(mapping)

,ala_species_name,expected_project_label,notes,normalised_ala_label,mapping_matches_expected
0,Australian Magpie,australian_magpie,NaN,australian_magpie,True
1,Laughing Kookaburra,laughing_kookaburra,NaN,laughing_kookaburra,True
2,Rainbow-Lorikeet,rainbow_lorikeet,NaN,rainbow_lorikeet,True
3,Sulphur Crested Cockatoo,sulphur_crested_cockatoo,NaN,sulphur_crested_cockatoo,True
4,Superb Fairy Wren,superb_fairy_wren,NaN,superb_fairy_wren,True
5,Willie wagtail,willie_wagtail,NaN,willie_wagtail,True
6,Crimson_Rosella,crimson_rosella,NaN,crimson_rosella,True
7,Eastern Rosella,eastern_rosella,NaN,eastern_rosella,True


## Step 6: Scan ALA folders

In [6]:
ala_records = []

for folder in sorted([p for p in ala_dir.iterdir() if p.is_dir()]):
    normalised_folder_label = normalise_label(folder.name)
    files = [p for p in folder.iterdir() if p.is_file()]
    audio_files = [p for p in files if p.suffix.lower() in [".wav", ".mp3"]]

    ala_records.append({
        "ala_folder_name": folder.name,
        "normalised_folder_label": normalised_folder_label,
        "total_files": len(files),
        "audio_files": len(audio_files),
        "non_audio_files": len(files) - len(audio_files)
    })

ala_summary = pd.DataFrame(ala_records)
display(ala_summary)

,ala_folder_name,normalised_folder_label,total_files,audio_files,non_audio_files
0,Australian Magpie,australian_magpie,6,6,0
1,Crimson_Rosella,crimson_rosella,6,6,0
2,Eastern Rosella,eastern_rosella,6,6,0
3,Laughing Kookaburra,laughing_kookaburra,6,6,0
4,Rainbow-Lorikeet,rainbow_lorikeet,6,6,0
5,Sulphur Crested Cockatoo,sulphur_crested_cockatoo,6,6,0
6,Superb Fairy Wren,superb_fairy_wren,6,6,0
7,Willie wagtail,willie_wagtail,6,6,0


## Step 7: Cross-check ALA labels with main manifest

In [7]:
manifest_labels = set(manifest["species_label"].dropna().astype(str))
expected_labels = set(mapping["expected_project_label"].dropna().astype(str))
ala_folder_labels = set(ala_summary["normalised_folder_label"].dropna().astype(str))

label_check = []
for label in sorted(ala_folder_labels):
    label_check.append({
        "ala_normalised_label": label,
        "exists_in_main_manifest": label in manifest_labels,
        "exists_in_expected_mapping": label in expected_labels
    })

label_check_df = pd.DataFrame(label_check)
display(label_check_df)

,ala_normalised_label,exists_in_main_manifest,exists_in_expected_mapping
0,australian_magpie,True,True
1,crimson_rosella,True,True
2,eastern_rosella,True,True
3,laughing_kookaburra,True,True
4,rainbow_lorikeet,True,True
5,sulphur_crested_cockatoo,True,True
6,superb_fairy_wren,True,True
7,willie_wagtail,True,True


## Step 8: Identify mapping issues

In [8]:
issues = []

# ALA folder labels missing from manifest
for _, row in label_check_df.iterrows():
    if not row["exists_in_main_manifest"]:
        issues.append({
            "issue_type": "ALA label not found in main manifest",
            "ala_label": row["ala_normalised_label"],
            "details": "The ALA folder normalises to a label that does not appear in the main manifest."
        })
    if not row["exists_in_expected_mapping"]:
        issues.append({
            "issue_type": "ALA label not found in expected mapping",
            "ala_label": row["ala_normalised_label"],
            "details": "The ALA folder normalises to a label that is not listed in the expected mapping file."
        })

# Expected mapping rows where normalised ALA name does not match project label
for _, row in mapping.iterrows():
    if not row["mapping_matches_expected"]:
        issues.append({
            "issue_type": "ALA naming drift",
            "ala_label": row["normalised_ala_label"],
            "details": f"ALA name '{row['ala_species_name']}' normalises to '{row['normalised_ala_label']}', expected project label is '{row['expected_project_label']}'."
        })

issues_df = pd.DataFrame(issues)
display(issues_df if not issues_df.empty else pd.DataFrame([{"status": "No mapping issues found"}]))

,status
0,No mapping issues found


## Step 9: Validate folder and file format compatibility

In [9]:
format_rows = []
allowed_extensions = {".wav", ".mp3"}

for folder in sorted([p for p in ala_dir.iterdir() if p.is_dir()]):
    for file in folder.iterdir():
        if file.is_file():
            format_rows.append({
                "ala_folder": folder.name,
                "file_name": file.name,
                "extension": file.suffix.lower(),
                "is_supported_audio": file.suffix.lower() in allowed_extensions,
                "file_size_bytes": file.stat().st_size,
                "is_empty_file": file.stat().st_size == 0
            })

format_report = pd.DataFrame(format_rows)
display(format_report.head())

format_issues = format_report[
    (~format_report["is_supported_audio"]) |
    (format_report["is_empty_file"])
]
display(format_issues if not format_issues.empty else pd.DataFrame([{"status": "No format issues found"}]))

,ala_folder,file_name,extension,is_supported_audio,file_size_bytes,is_empty_file
0,Australian Magpie,ALA_Australian_Magpie_006.wav,.wav,True,47838,False
1,Australian Magpie,ALA_Australian_Magpie_005.wav,.wav,True,61318,False
2,Australian Magpie,ALA_Australian_Magpie_001.wav,.wav,True,74338,False
3,Australian Magpie,ALA_Australian_Magpie_002.wav,.wav,True,69628,False
4,Australian Magpie,ALA_Australian_Magpie_003.wav,.wav,True,81006,False


,status
0,No format issues found


## Step 10: Confirm compatibility with Engine-style directory structure

In [10]:
structure_check = pd.DataFrame({
    "check": [
        "ALA portion is folder-per-species",
        "Folder labels can be normalised to project labels",
        "Files are stored inside species folders",
        "Supported file extensions are used",
        "Main manifest exists",
        "ALA expected mapping exists"
    ],
    "status": [
        ala_dir.exists(),
        ala_summary["normalised_folder_label"].notna().all(),
        (ala_summary["total_files"] > 0).all(),
        format_issues.empty,
        manifest_path.exists(),
        mapping_path.exists()
    ]
})
display(structure_check)

,check,status
0,ALA portion is folder-per-species,True
1,Folder labels can be normalised to project labels,True
2,Files are stored inside species folders,True
3,Supported file extensions are used,True
4,Main manifest exists,True
5,ALA expected mapping exists,True


## Step 11: Create recommended corrections

In [11]:
corrections = mapping.copy()
corrections["recommended_folder_name"] = corrections["expected_project_label"]
corrections["action_required"] = corrections.apply(
    lambda r: "Rename/align ALA label to expected project label" if not r["mapping_matches_expected"] else "No change required",
    axis=1
)
display(corrections[["ala_species_name", "normalised_ala_label", "expected_project_label", "recommended_folder_name", "action_required"]])

,ala_species_name,normalised_ala_label,expected_project_label,recommended_folder_name,action_required
0,Australian Magpie,australian_magpie,australian_magpie,australian_magpie,No change required
1,Laughing Kookaburra,laughing_kookaburra,laughing_kookaburra,laughing_kookaburra,No change required
2,Rainbow-Lorikeet,rainbow_lorikeet,rainbow_lorikeet,rainbow_lorikeet,No change required
3,Sulphur Crested Cockatoo,sulphur_crested_cockatoo,sulphur_crested_cockatoo,sulphur_crested_cockatoo,No change required
4,Superb Fairy Wren,superb_fairy_wren,superb_fairy_wren,superb_fairy_wren,No change required
5,Willie wagtail,willie_wagtail,willie_wagtail,willie_wagtail,No change required
6,Crimson_Rosella,crimson_rosella,crimson_rosella,crimson_rosella,No change required
7,Eastern Rosella,eastern_rosella,eastern_rosella,eastern_rosella,No change required


## Step 12: Save reports

In [12]:
output_dir = Path("/content/task5_outputs")
output_dir.mkdir(exist_ok=True)

ala_summary.to_csv(output_dir / "ala_folder_summary.csv", index=False)
mapping.to_csv(output_dir / "ala_mapping_validation.csv", index=False)
label_check_df.to_csv(output_dir / "ala_manifest_label_check.csv", index=False)
issues_df.to_csv(output_dir / "species_mapping_issues.csv", index=False)
format_report.to_csv(output_dir / "format_validation_report.csv", index=False)
corrections.to_csv(output_dir / "recommended_mapping_corrections.csv", index=False)
structure_check.to_csv(output_dir / "structure_compatibility_check.csv", index=False)

print("Saved reports in:", output_dir)
print(os.listdir(output_dir))

Saved reports in: /content/task5_outputs
['species_mapping_issues.csv', 'structure_compatibility_check.csv', 'ala_manifest_label_check.csv', 'format_validation_report.csv', 'recommended_mapping_corrections.csv', 'ala_mapping_validation.csv', 'ala_folder_summary.csv']


## Step 13: Generate final alignment confirmation note

In [13]:
total_ala_species = len(ala_summary)
mapping_issue_count = len(issues_df)
format_issue_count = len(format_issues)
compatible = mapping_issue_count == 0 and format_issue_count == 0 and structure_check["status"].all()

note = f"""
Task 5: ALA Species and Label Mapping Validation

The ALA portion of the merged dataset was reviewed against the main species manifest and expected ALA mapping file.

Summary:
- Total ALA species folders reviewed: {total_ala_species}
- Mapping issues found: {mapping_issue_count}
- File format issues found: {format_issue_count}
- Folder-per-species structure present: {ala_dir.exists()}
- Main manifest available: {manifest_path.exists()}
- Expected mapping file available: {mapping_path.exists()}

Result:
{'The ALA portion is compatible with the current training/data-loading structure.' if compatible else 'Some ALA naming or format issues were identified and listed in the output reports.'}

Actions:
- Species names were normalised into project-style labels.
- ALA labels were cross-checked against the main manifest.
- File formats were checked for compatibility with the pipeline.
- Mapping inconsistencies were documented in species_mapping_issues.csv.
- Recommended corrections were saved in recommended_mapping_corrections.csv.
"""

note_path = output_dir / "final_alignment_confirmation.txt"
note_path.write_text(note)

print(note)


Task 5: ALA Species and Label Mapping Validation

The ALA portion of the merged dataset was reviewed against the main species manifest and expected ALA mapping file.

Summary:
- Total ALA species folders reviewed: 8
- Mapping issues found: 0
- File format issues found: 0
- Folder-per-species structure present: True
- Main manifest available: True
- Expected mapping file available: True

Result:
The ALA portion is compatible with the current training/data-loading structure.

Actions:
- Species names were normalised into project-style labels.
- ALA labels were cross-checked against the main manifest.
- File formats were checked for compatibility with the pipeline.
- Mapping inconsistencies were documented in species_mapping_issues.csv.
- Recommended corrections were saved in recommended_mapping_corrections.csv.



## Step 14: Zip and download outputs

In [14]:
shutil.make_archive("/content/task5_alignment_outputs", "zip", output_dir)

from google.colab import files
files.download("/content/task5_alignment_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>